# Comparatif des modeles sur les donnees du projet

Ce notebook genere un tableau comparatif similaire a ton exemple, calcule sur les donnees du projet.

Modeles compares :
- LR (grand volume)
- LR (petit volume)
- Tous les checkpoints Transformers detectes automatiquement (ex: DistilBERT, Mental-BERT, autres)

Metrices : recall (depressed), precision (depressed), accuracy, taille train, temps entrainement, interpretabilite, vitesse inference.

Le tableau se met a jour automatiquement: si un nouveau modele est ajoute dans models/fine_tuned, une nouvelle colonne est creee.

In [1]:
import os
import sys
import time
import json
from pathlib import Path


def _find_project_root() -> Path:
    """Remonte depuis le notebook (ou CWD) jusqu'a la racine du projet."""
    markers = ('src', 'requirements.txt', 'pytest.ini', 'pyproject.toml')
    # VS Code Jupyter expose le chemin reel du notebook dans __vsc_ipynb_file__
    try:
        start = Path(__vsc_ipynb_file__).parent  # noqa: F821
    except NameError:
        start = Path.cwd()
    for candidate in [start, *start.parents]:
        if any((candidate / m).exists() for m in markers):
            return candidate
    return start  # fallback


ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

from src.training.preprocess import clean_text, load_kaggle_depression

pd.set_option('display.max_colwidth', None)
pd.set_option('display.precision', 3)

print('ROOT          :', ROOT)
print('Imports OK')

ROOT          : /home/fmoncaut/healthcare/mental-health-signal-detector
Imports OK


In [2]:
# Configuration des chemins (relatifs a ROOT, pas au CWD du notebook)
DATA_CANDIDATES = [
    ROOT / 'data/raw',
    Path('/content/drive/MyDrive/mh_detector/data'),
    Path('/content/mh_detector/data'),
]

def first_existing_path(paths):
    for p in paths:
        if p.exists():
            return p
    return None

DATA_DIR = first_existing_path(DATA_CANDIDATES)
if DATA_DIR is None:
    raise FileNotFoundError('Aucun dossier data trouve. Ajuste DATA_CANDIDATES.')

KAGGLE_PATH = DATA_DIR / 'reddit_depression_dataset.csv'
ERISK25_PATH = DATA_DIR / 'erisk25'

print('DATA_DIR      :', DATA_DIR)
print('KAGGLE_PATH   :', KAGGLE_PATH, '| exists =', KAGGLE_PATH.exists())
print('ERISK25_PATH  :', ERISK25_PATH, '| exists =', ERISK25_PATH.exists())

DATA_DIR      : /home/fmoncaut/healthcare/mental-health-signal-detector/data/raw
KAGGLE_PATH   : /home/fmoncaut/healthcare/mental-health-signal-detector/data/raw/reddit_depression_dataset.csv | exists = True
ERISK25_PATH  : /home/fmoncaut/healthcare/mental-health-signal-detector/data/raw/erisk25 | exists = True


In [3]:
# Chargeur eRisk25 robuste (evite les erreurs I/O type errno 107 sur Drive)
def load_erisk25_robust(data_path: Path, retries: int = 3) -> pd.DataFrame:
    json_dir = data_path / 'final-eriskt2-dataset-with-ground-truth' / 'all_combined'
    files = sorted(json_dir.glob('subject_*.json'))
    if not files:
        raise FileNotFoundError(f'Aucun fichier subject_*.json dans {json_dir}')

    rows = []
    skipped_os = 0
    skipped_json = 0

    for filepath in files:
        posts = None
        for attempt in range(1, retries + 1):
            try:
                with filepath.open('r', encoding='utf-8') as f:
                    posts = json.load(f)
                break
            except OSError:
                if attempt < retries:
                    time.sleep(0.6 * attempt)
                    continue
                skipped_os += 1
                posts = None
                break
            except json.JSONDecodeError:
                skipped_json += 1
                posts = None
                break

        if not posts:
            continue

        for post in posts:
            sub = post.get('submission', {})
            target = sub.get('target')
            if target is None:
                continue
            text = ((sub.get('title') or '') + ' ' + (sub.get('body') or '')).strip()
            if len(text) > 20:
                rows.append({'text': text, 'label': int(bool(target))})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'eRisk25 vide. skipped_os={skipped_os}, skipped_json={skipped_json}')

    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10].drop_duplicates(subset=['text']).reset_index(drop=True)

    print(f'eRisk25 : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    print(f'Fichiers ignores -> OSError: {skipped_os}, JSON: {skipped_json}')
    return df

In [4]:
# Construction du dataset de travail
# On separe les splits Kaggle et eRisk25 AVANT de fusionner
# pour pouvoir evaluer sur Kaggle-only (comparaison avec Stan) ET sur le mix complet.

kaggle_train, kaggle_test, erisk_train, erisk_test = None, None, None, None

if KAGGLE_PATH.exists():
    kaggle_df = load_kaggle_depression(KAGGLE_PATH, max_samples=100_000)
    print(f'Kaggle : {len(kaggle_df)} lignes | {kaggle_df["label"].value_counts().to_dict()}')
    kaggle_train, kaggle_test = train_test_split(
        kaggle_df, test_size=0.2, random_state=42, stratify=kaggle_df['label']
    )

if ERISK25_PATH.exists():
    erisk_df = load_erisk25_robust(ERISK25_PATH)
    erisk_train, erisk_test = train_test_split(
        erisk_df, test_size=0.2, random_state=42, stratify=erisk_df['label']
    )

# Dataset mixte (train/test complets)
frames_train = [f for f in [kaggle_train, erisk_train] if f is not None]
frames_test  = [f for f in [kaggle_test,  erisk_test]  if f is not None]
if not frames_train:
    raise RuntimeError('Aucune source chargee.')

train_df = pd.concat(frames_train, ignore_index=True).drop_duplicates(subset=['text']).reset_index(drop=True)
test_df  = pd.concat(frames_test,  ignore_index=True).drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Dataset MIXTE  -> train: {len(train_df)} | test: {len(test_df)}')

if kaggle_test is not None:
    print(f'Dataset KAGGLE -> train: {len(kaggle_train)} | test: {len(kaggle_test)}')


In [5]:
def format_train_size(n: int) -> str:
    if n >= 1_000_000:
        return f'{n/1_000_000:.1f}M'
    if n >= 1_000:
        return f'{n/1_000:.0f}K'
    return str(n)

def speed_bucket(samples_per_sec: float) -> str:
    if samples_per_sec >= 2000:
        return 'Very fast'
    if samples_per_sec >= 500:
        return 'Fast'
    if samples_per_sec >= 100:
        return 'Medium'
    return 'Slow'

_MODEL_TYPE_LABELS = {
    'distilbert':  'DistilBERT',
    'bert':        'BERT',
    'roberta':     'RoBERTa',
    'albert':      'ALBERT',
    'deberta':     'DeBERTa',
    'xlnet':       'XLNet',
    'gpt2':        'GPT-2',
    'mental-bert': 'Mental-BERT',
}

def _pretty_model_name(model_dir: Path) -> str:
    """Lit config.json pour deduire un label lisible (ex: DistilBERT)."""
    cfg_path = model_dir / 'config.json'
    try:
        with cfg_path.open('r') as f:
            cfg = json.load(f)
        raw = cfg.get('model_type', model_dir.name)
        return _MODEL_TYPE_LABELS.get(raw.lower(), raw.capitalize())
    except Exception:
        return model_dir.name.replace('_', '-').capitalize()

def discover_transformer_models(search_roots: list[Path]) -> list[tuple[str, Path]]:
    discovered = []
    seen = set()

    for root in search_roots:
        if not root.exists():
            continue

        for cfg in root.rglob('config.json'):
            model_dir = cfg.parent
            has_tokenizer = (model_dir / 'tokenizer_config.json').exists()
            has_weights = (model_dir / 'model.safetensors').exists() or (model_dir / 'pytorch_model.bin').exists()
            if not (has_tokenizer and has_weights):
                continue

            key = str(model_dir.resolve())
            if key in seen:
                continue
            seen.add(key)

            display_name = _pretty_model_name(model_dir)
            discovered.append((display_name, model_dir))

    discovered.sort(key=lambda x: x[0].lower())
    return discovered

def eval_lr(train_part: pd.DataFrame, test_part: pd.DataFrame, name: str) -> dict:
    start_train = time.perf_counter()
    model = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced'))
    ])
    model.fit(train_part['text'], train_part['label'])
    train_time = time.perf_counter() - start_train

    y_true = test_part['label']
    y_pred = model.predict(test_part['text'])

    sample_texts = test_part['text'].iloc[: min(500, len(test_part))]
    start_inf = time.perf_counter()
    _ = model.predict(sample_texts)
    inf_time = max(time.perf_counter() - start_inf, 1e-9)
    sps = len(sample_texts) / inf_time

    return {
        'Model': name,
        'Recall (depressed)': recall_score(y_true, y_pred, pos_label=1),
        'Precision (depressed)': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Training data': format_train_size(len(train_part)),
        'Training time': f'{train_time:.1f}s',
        'Interpretable': 'Yes',
        'Inference speed': speed_bucket(sps),
    }

def eval_transformer_from_path(
    model_path: Path,
    train_size: int,
    test_part: pd.DataFrame,
    name: str
) -> dict:
    if not model_path.exists():
        return {
            'Model': name,
            'Recall (depressed)': np.nan,
            'Precision (depressed)': np.nan,
            'Accuracy': np.nan,
            'Training data': format_train_size(train_size),
            'Training time': 'N/A (modele absent)',
            'Interpretable': 'No',
            'Inference speed': 'N/A',
        }

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    texts = test_part['text'].tolist()
    labels = test_part['label'].to_numpy()

    preds = []
    batch_size = 32

    start_inf = time.perf_counter()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        preds.extend(torch.argmax(logits, dim=-1).cpu().numpy().tolist())
    inf_time = max(time.perf_counter() - start_inf, 1e-9)

    sps = len(texts) / inf_time

    return {
        'Model': name,
        'Recall (depressed)': recall_score(labels, preds, pos_label=1),
        'Precision (depressed)': precision_score(labels, preds, pos_label=1, zero_division=0),
        'Accuracy': accuracy_score(labels, preds),
        'Training data': format_train_size(train_size),
        'Training time': 'Voir log entrainement',
        'Interpretable': 'No',
        'Inference speed': speed_bucket(sps),
    }


In [6]:
SMALL_TRAIN_SIZE = 21_000

MODEL_SEARCH_ROOTS = [
    ROOT / 'models/fine_tuned',
    ROOT / 'models',
]
detected_models = discover_transformer_models(MODEL_SEARCH_ROOTS)
print(f'Modeles Transformers detectes: {len(detected_models)}')
for model_name, model_path in detected_models:
    print(' -', model_name, '->', model_path)

def run_all_evals(train_full, test_full, train_kaggle, test_kaggle, small_size):
    """Retourne (results_mixed, results_kaggle)."""
    # Petit subset Kaggle pour LR 21K
    if train_kaggle is not None and len(train_kaggle) > small_size:
        train_small_kag, _ = train_test_split(
            train_kaggle, train_size=small_size, random_state=42, stratify=train_kaggle['label']
        )
    elif train_kaggle is not None:
        train_small_kag = train_kaggle
    else:
        train_small_kag = None

    # Petit subset Mixed pour LR 21K
    if len(train_full) > small_size:
        train_small_mix, _ = train_test_split(
            train_full, train_size=small_size, random_state=42, stratify=train_full['label']
        )
    else:
        train_small_mix = train_full

    # ── Evaluations sur dataset MIXTE ──
    res_mix = []
    res_mix.append(eval_lr(train_full,      test_full, name=f'LR ({format_train_size(len(train_full))})'))
    res_mix.append(eval_lr(train_small_mix, test_full, name=f'LR ({format_train_size(len(train_small_mix))})'))
    for model_name, model_path in detected_models:
        res_mix.append(eval_transformer_from_path(
            model_path=model_path,
            train_size=small_size,
            test_part=test_full,
            name=f'{model_name} ({format_train_size(small_size)})',
        ))

    # ── Evaluations sur Kaggle UNIQUEMENT (comparaison Stan) ──
    res_kag = []
    if train_kaggle is not None:
        res_kag.append(eval_lr(train_kaggle,     test_kaggle, name=f'LR ({format_train_size(len(train_kaggle))})'))
        res_kag.append(eval_lr(train_small_kag,  test_kaggle, name=f'LR ({format_train_size(len(train_small_kag))})'))
        for model_name, model_path in detected_models:
            res_kag.append(eval_transformer_from_path(
                model_path=model_path,
                train_size=small_size,
                test_part=test_kaggle,
                name=f'{model_name} ({format_train_size(small_size)})',
            ))

    return res_mix, res_kag

results_mixed, results_kaggle = run_all_evals(
    train_df, test_df,
    kaggle_train, kaggle_test,
    SMALL_TRAIN_SIZE
)
print('Evaluations terminees.')


In [7]:
def make_table(results: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(results).set_index('Model').T
    for row in ['Recall (depressed)', 'Precision (depressed)', 'Accuracy']:
        df.loc[row] = pd.to_numeric(df.loc[row], errors='coerce').round(3)
    return df

reports_dir = ROOT / 'reports'
reports_dir.mkdir(exist_ok=True)

# Tableau 1 — Dataset KAGGLE uniquement (comparaison directe avec Stan)
if results_kaggle:
    table_kaggle = make_table(results_kaggle)
    print('=' * 70)
    print('TABLEAU 1 — Kaggle uniquement (comparaison avec Stan)')
    print('=' * 70)
    display(table_kaggle)
    out1 = reports_dir / 'model_comparison_kaggle.csv'
    table_kaggle.to_csv(out1)
    print('Exporte ->', out1)

# Tableau 2 — Dataset MIXTE (Kaggle + eRisk25) — generalisation reelle
table_mixed = make_table(results_mixed)
print()
print('=' * 70)
print('TABLEAU 2 — Mix Kaggle + eRisk25 (generalisation reelle)')
print('=' * 70)
display(table_mixed)
out2 = reports_dir / 'model_comparison_table.csv'
table_mixed.to_csv(out2)
print('Exporte ->', out2)


## Notes

- Le tableau ajoute automatiquement les checkpoints Transformers trouves dans models/fine_tuned et models.
- Si aucun checkpoint valide n est detecte, seules les colonnes LR sont affichees.
- Pour comparer un nouveau modele, il suffit de sauvegarder son dossier (config.json + tokenizer_config.json + poids) dans models/fine_tuned puis relancer les cellules 6, 7 et 8.